In [ ]:
!cp "/content/drive/MyDrive/Hate Speech LLMs/ARR Data/Human Validation/humanData.xlsx" .

In [ ]:
import numpy as np
import pandas as pd
import ast
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score

In [ ]:
df = pd.read_excel("humanData.xlsx")

In [ ]:
annotator1_col = 'Hop_2_Target_Targeted_Group (A1)'
annotator2_col = 'Hop_2_Target_Targeted_Group (A2)'

In [ ]:
df = df[df["Hop_1_Triage (A1)"] != "بی‌ضرر"]
df = df[df["Hop_1_Triage (A2)"] != "بی‌ضرر"]
print(len(df))

68


In [ ]:
annotator1Groups = []
for annotator1Group in df[annotator1_col]:
  annotator1Groups.extend(ast.literal_eval(annotator1Group))
set(annotator1Groups)

{'other',
 'دانشجویان، کودکان و نوجوانان',
 'دشمنان خارجی',
 'زنان ایرانی',
 'شرکت\u200cها، سازمان\u200cها و کارکنان آن\u200cها',
 'فعالان حقوق بشر، رسانه\u200cای و دانشگاهی',
 'مخالفان سیاسی داخلی',
 'مردان ایرانی',
 'مقامات و سیاستمداران ایران',
 'مهاجران و اقلیت\u200cهای قومی/مذهبی',
 'نیروهای امنیتی و نظامی ایران',
 'هنرمندان، بازیگران و سلبریتی\u200cها'}

In [ ]:
annotator2Groups = []
for annotator2Group in df[annotator2_col]:
  annotator2Groups.extend(ast.literal_eval(annotator2Group))
set(annotator2Groups)

{'other',
 'دشمنان خارجی',
 'روحانیون و رهبران مذهبی',
 'زنان ایرانی',
 'شرکت\u200cها، سازمان\u200cها و کارکنان آن\u200cها',
 'فعالان حقوق بشر، رسانه\u200cای و دانشگاهی',
 'مخالفان سیاسی داخلی',
 'مردان ایرانی',
 'مقامات و سیاستمداران ایران',
 'مهاجران و اقلیت\u200cهای قومی/مذهبی',
 'نیروهای امنیتی و نظامی ایران',
 'هنرمندان، بازیگران و سلبریتی\u200cها'}

In [ ]:
len(set(annotator1Groups))

12

In [ ]:
len(set(annotator2Groups))

12

In [ ]:
df[annotator1_col] = df[annotator1_col].apply(ast.literal_eval)
df[annotator2_col] = df[annotator2_col].apply(ast.literal_eval)

In [ ]:
# Remove "other" from both columns
df[annotator1_col] = df[annotator1_col].apply(
    lambda x: [label for label in x if label != 'other']
)

df[annotator2_col] = df[annotator2_col].apply(
    lambda x: [label for label in x if label != 'other']
)

# Remove rows where either side becomes empty
df_eval = df[
    df[annotator1_col].apply(len).gt(0) &
    df[annotator2_col].apply(len).gt(0)
].copy()

In [ ]:
mlb = MultiLabelBinarizer()

mlb.fit(list(df_eval[annotator1_col]) + list(df_eval[annotator2_col]))

y_annotator1 = mlb.transform(df_eval[annotator1_col])
y_annotator2 = mlb.transform(df_eval[annotator2_col])

In [ ]:
f1_micro = f1_score(y_annotator1, y_annotator2, average='micro')
f1_macro = f1_score(y_annotator1, y_annotator2, average='macro')

print(f"Micro F1: {f1_micro:.4f}")
print(f"Macro F1: {f1_macro:.4f}")

Micro F1: 0.8205
Macro F1: 0.6934
